# FD02: Cost-sensitive XGBoost
Source for the FD02 notebook. Convert with `jupytext --to ipynb` and run on Kaggle, or set `FRAUD_DATA_PATH` locally.
Dataset: Credit Card Fraud Detection (`mlg-ulb/creditcardfraud`), time-ordered split 70/15/15.
Train on the earliest 70% of time, tune on the next 15%, report the last 15% once.
Question: how much does a cost-sensitive tree model improve on the FD01 logistic baseline?

In [1]:
from pathlib import Path

import pandas as pd

from fraud_xgb import plots
from fraud_xgb.data import load_creditcard, time_split
from fraud_xgb.export import build_score_export, write_score_export
from fraud_xgb.metrics import brier, confusion_at_threshold, metric_summary
from fraud_xgb.model import (
    FEATURES,
    add_features,
    build_baseline,
    build_tuned,
    gain_importance,
    scale_pos_weight,
    search,
)

IN_NOTEBOOKS = Path.cwd().name == "notebooks"
FIG_DIR = Path("../reports/figures") if IN_NOTEBOOKS else Path("reports/figures")
REPORT_DIR = Path("../reports") if IN_NOTEBOOKS else Path("reports")
DATA_DIR = Path("../data") if IN_NOTEBOOKS else Path("data")
ALERT_RATE = 0.01
N_SEARCH_CONFIGS = 30
# Record the FD01 logistic test PR-AUC from the FD01 notebook output before the comparison is read.
FD01_TEST_PR_AUC = None
DATASET = "creditcardfraud"
BASELINE_LABEL = "XGBoost baseline (scale_pos_weight, fixed params)"
TUNED_LABEL = f"XGBoost tuned ({N_SEARCH_CONFIGS} random configs)"

In [2]:
df = add_features(load_creditcard())
split = time_split(df)
X, y = df[FEATURES], df["Class"]
print(f"rows: {len(df):,}  frauds: {int(y.sum())}")
print(f"train end (Time): {split.train_end:,.0f}s  val end: {split.val_end:,.0f}s")
for name, mask in (("train", split.train), ("val", split.val), ("test", split.test)):
    print(f"{name}: {int(mask.sum()):,} rows, {int(y[mask].sum())} frauds")

rows: 284,807  frauds: 492
train end (Time): 132,929s  val end: 151,328s
train: 199,368 rows, 384 frauds
val: 42,718 rows, 56 frauds
test: 42,721 rows, 52 frauds


In [3]:
spw = scale_pos_weight(y[split.train])
print(f"scale_pos_weight (train only): {spw:.3f}")

scale_pos_weight (train only): 518.188


## Baseline on validation (fixed parameters, no tuning)

In [4]:
baseline = build_baseline(spw).fit(X[split.train], y[split.train])
baseline_val = baseline.predict_proba(X[split.val])[:, 1]
print(metric_summary(y[split.val], baseline_val, ALERT_RATE))

{'pr_auc': 0.8440919916593012, 'roc_auc': 0.9829724679708272, 'recall_at_1.00%_alerts': 0.9285714285714286}


## Random search on validation PR-AUC only
Each config is fit on the training window and scored on validation. The test window is not touched.

In [5]:
best_params, trials = search(
    X[split.train], y[split.train], X[split.val], y[split.val], n_iter=N_SEARCH_CONFIGS
)
print(trials.sort_values("val_pr_auc", ascending=False).head(5).to_string(index=False))
print(f"chosen config: {best_params}")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
trials.to_csv(REPORT_DIR / "fd02_search_trials.csv", index=False)

 trial  colsample_bytree  learning_rate  max_depth  min_child_weight  n_estimators  reg_lambda  subsample  val_pr_auc
     4          0.716858       0.034766          8                 3           463    1.067748   0.836966    0.871436
     8          0.770062       0.020284          6                 6           576    8.692992   0.910053    0.863169
     1          0.639990       0.047683          7                 4           459    2.607025   0.608234    0.862789
    20          0.648835       0.033597          8                 1           484    0.285674   0.770843    0.862757
    19          0.718253       0.014316          6                 4           385    5.841986   0.729738    0.862198
chosen config: {'max_depth': 8, 'learning_rate': 0.03476649150592621, 'n_estimators': 463, 'subsample': 0.836965827544817, 'colsample_bytree': 0.7168578594140873, 'min_child_weight': 3, 'reg_lambda': 1.0677482709481354}


## Refit the chosen config on training rows only

In [6]:
tuned = build_tuned(best_params, spw).fit(X[split.train], y[split.train])
tuned_val = tuned.predict_proba(X[split.val])[:, 1]
print(metric_summary(y[split.val], tuned_val, ALERT_RATE))

{'pr_auc': 0.8714364145749088, 'roc_auc': 0.9881811849956804, 'recall_at_1.00%_alerts': 0.9107142857142857}


## Test window (reported once)

In [7]:
test_y = y[split.test]
baseline_test = baseline.predict_proba(X[split.test])[:, 1]
tuned_test = tuned.predict_proba(X[split.test])[:, 1]

results = {}
for label, probs in ((BASELINE_LABEL, baseline_test), (TUNED_LABEL, tuned_test)):
    summary = metric_summary(test_y, probs, ALERT_RATE)
    summary["brier"] = brier(test_y, probs)
    results[label] = summary
    print(label)
    print(summary)
    print(confusion_at_threshold(test_y, probs, threshold=0.5))

XGBoost baseline (scale_pos_weight, fixed params)
{'pr_auc': 0.7646833518181771, 'roc_auc': 0.9824030056048618, 'recall_at_1.00%_alerts': 0.8269230769230769, 'brier': 0.0004424000217113644}
{'tn': 42661, 'fp': 8, 'fn': 13, 'tp': 39}
XGBoost tuned (30 random configs)
{'pr_auc': 0.7717845409546235, 'roc_auc': 0.9793283540383307, 'recall_at_1.00%_alerts': 0.8461538461538461, 'brier': 0.0004525614494923502}
{'tn': 42662, 'fp': 7, 'fn': 14, 'tp': 38}


In [8]:
test_pr_auc = results[TUNED_LABEL]["pr_auc"]
if FD01_TEST_PR_AUC is None:
    print("FD01 test PR-AUC not recorded yet; the decision needs that value.")
else:
    print(f"tuned XGBoost minus FD01 logistic test PR-AUC: {test_pr_auc - FD01_TEST_PR_AUC:+.4f}")

FD01 test PR-AUC not recorded yet; the decision needs that value.


## Figures

In [9]:
title_suffix = f"{DATASET}, test split"
plots.plot_pr_curves(
    test_y,
    {"Baseline": baseline_test, "Tuned": tuned_test},
    f"PR curve, {title_suffix}, baseline vs tuned",
    FIG_DIR / "fd02_pr_curve.png",
)
importance = gain_importance(tuned)
plots.plot_gain_importance(
    importance,
    f"Feature importance by gain, {title_suffix}, tuned XGBoost",
    FIG_DIR / "fd02_feature_importance.png",
)
plots.plot_calibration(
    test_y,
    tuned_test,
    f"Reliability diagram, {title_suffix}, tuned XGBoost",
    FIG_DIR / "fd02_calibration.png",
)
importance.to_csv(REPORT_DIR / "fd02_feature_importance.csv", index=False)
print(importance.head(10))

  feature         gain  gain_share
0     V14  3774.235596    0.355657
1     V10  1746.533081    0.164581
2     V12   894.955933    0.084334
3     V17   662.539490    0.062433
4      V4   572.298218    0.053929
5      V8   226.465805    0.021341
6      V3   201.858932    0.019022
7     V19   169.474228    0.015970
8     V20   156.513962    0.014749
9     V11   146.098785    0.013767


## Exports for FD05
`fd02_scores.csv` holds validation and test rows only, scored by the tuned model.

In [10]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
tuned.get_booster().save_model(str(DATA_DIR / "fd02_xgb_tuned.json"))
export = build_score_export(df, split, tuned)
write_score_export(export, DATA_DIR / "fd02_scores.csv")
print(export["split"].value_counts())

split
test    42721
val     42718
Name: count, dtype: int64
